## Protocole d'évaluation du projet African Crop Yield Intelligence

Ce notebook définit la manière dont les modèles de rendement seront évalués. Il correspond à la conception du plan de test de la phase Modeling de CRISP-DM et remplit la troisième condition de la décision issue du Data Understanding. Il fait suite à `03_data_preparation.ipynb`.

**Exigences**

- Aucun lieu ne doit figurer à la fois dans l'entraînement et dans l'évaluation. Les parcelles d'une même grappe partagent leur sol et leur climat, les ménages du panel reviennent d'une vague à l'autre, et une région de niveau 1 recouvre ses unités de niveau 2.
- La capacité à prédire une période future et la capacité à prédire un lieu jamais observé doivent être mesurées séparément.
- Les ensembles de test ne servent qu'une fois, pour évaluer les modèles retenus à l'issue de la modélisation. Toute sélection de modèle ou d'hyperparamètres s'appuie sur la validation croisée du développement.
- Tout modèle doit être comparé à une référence simple, dépourvue de variable explicative.

**Méthode**

Les partitions, les métriques et les références sont implémentées sous forme de fonctions testées dans `src/evaluation/`. L'affectation d'un groupe à une partition dépend uniquement d'une empreinte SHA-256 de son identifiant. Elle est donc reproductible, indépendante de l'ordre des lignes et stable lorsque des données sont ajoutées.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.data.load import DOSSIER_TRAITE
from src.data.regional import zone_geographique
from src.evaluation.metriques import metriques, metriques_par, moyenne_macro
from src.evaluation.partitions import (
    ANNEE_COUPURE_REGIONAL,
    DEVELOPPEMENT,
    NOMBRE_PLIS,
    TEST_SPATIAL,
    TEST_SPATIO_TEMPOREL,
    TEST_TEMPOREL,
    partitionner_cropcut,
    partitionner_regional,
    partitionner_survey,
)
from src.evaluation.references import NIVEAUX_REGIONAL, NIVEAUX_SURVEY, reference_hierarchique

pd.set_option("display.max_columns", 40)
TESTS = [TEST_TEMPOREL, TEST_SPATIAL, TEST_SPATIO_TEMPOREL]
ORDRE = [DEVELOPPEMENT] + TESTS

### 1. Tests unitaires

La suite complète couvre les fonctions de préparation et celles du protocole d'évaluation.

In [2]:
execution = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=RACINE, capture_output=True, text=True,
)
print(execution.stderr.strip().splitlines()[-3:])

['Ran 46 tests in 0.361s', '', 'OK']


### 2. Construction des partitions

Les jeux produits par la Data Preparation sont chargés depuis `data/processed`. Chaque ligne reçoit un groupe, une partition et, dans le développement, un pli de validation croisée. Le croisement de deux critères définit quatre partitions.

| Partition | Lieu | Période |
|---|---|---|
| `developpement` | Groupe conservé | Passée |
| `test_temporel` | Groupe conservé | Future |
| `test_spatial` | Groupe tenu à l'écart | Passée |
| `test_spatio_temporel` | Groupe tenu à l'écart | Future |

Un groupe est tenu à l'écart lorsque l'empreinte de son identifiant tombe dans les 20 premiers centièmes. Le développement est réparti en 5 plis de validation croisée, chaque groupe appartenant à un seul pli.

In [3]:
regional = pd.read_csv(DOSSIER_TRAITE / "regional_modelisation.csv.gz", low_memory=False)
survey = pd.read_csv(DOSSIER_TRAITE / "survey_modelisation.csv.gz", low_memory=False)
cropcut = pd.read_csv(DOSSIER_TRAITE / "cropcut_validation.csv.gz", low_memory=False)

regional["zone"] = zone_geographique(regional)
regional = partitionner_regional(regional)
survey = partitionner_survey(survey)
cropcut = partitionner_cropcut(cropcut, survey)


def bilan_partitions(df, pays):
    effectifs = df["partition"].value_counts().reindex(ORDRE)
    return pd.DataFrame({
        "lignes": effectifs,
        "part": (effectifs / len(df)).round(3),
        "groupes": df.groupby("partition")["groupe"].nunique().reindex(ORDRE),
        "pays": df.groupby("partition")[pays].nunique().reindex(ORDRE),
    })


def controler_etancheite(df):
    ecarte = df["partition"].isin([TEST_SPATIAL, TEST_SPATIO_TEMPOREL])
    groupes_des_deux_cotes = int((ecarte.groupby(df["groupe"]).nunique() > 1).sum())
    plis_par_groupe = df.dropna(subset=["pli"]).groupby("groupe")["pli"].nunique().max()
    return {"groupes présents des deux côtés": groupes_des_deux_cotes, "plis maximum par groupe": int(plis_par_groupe)}

### 3. Partition de `regional`

Le groupe réunit, par composantes connexes, une région de niveau 1 et les unités de niveau 2 qui s'y rattachent. Un agrégat régional et ses composantes se trouvent ainsi du même côté, y compris lorsqu'une unité de niveau 2 est rattachée à plusieurs régions comme au Ghana. La période future regroupe les années de récolte postérieures ou égales à 2017.

In [4]:
display(bilan_partitions(regional, "L0_GID"))
print("Étanchéité :", controler_etancheite(regional))
print("Années du développement :", regional.loc[regional["partition"] == DEVELOPPEMENT, "harvest_year"].agg(["min", "max"]).tolist())
print("Lignes par pli :", regional["pli"].value_counts().sort_index().to_dict())

couverture = regional.groupby("country")["partition"].value_counts().unstack(fill_value=0).reindex(columns=ORDRE, fill_value=0)
sans_test_spatial = couverture.index[couverture[[TEST_SPATIAL, TEST_SPATIO_TEMPOREL]].sum(axis=1) == 0]
sans_futur = couverture.index[couverture[[TEST_TEMPOREL, TEST_SPATIO_TEMPOREL]].sum(axis=1) == 0]
print("\nPays sans région tenue à l'écart :", len(sans_test_spatial), list(sans_test_spatial))
print("Pays sans donnée à partir de", ANNEE_COUPURE_REGIONAL, ":", len(sans_futur))
print("Sources du test temporel :", regional.loc[regional["partition"] == TEST_TEMPOREL, "Source"].value_counts().to_dict())

,lignes,part,groupes,pays
partition,,,,
developpement,105724,0.725,539,45
test_temporel,13226,0.091,244,22
test_spatial,23817,0.163,130,39
test_spatio_temporel,3110,0.021,60,17


Étanchéité : {'groupes présents des deux côtés': 0, 'plis maximum par groupe': 1}
Années du développement : [1960, 2016]
Lignes par pli : {np.int64(0): 17347, np.int64(1): 25647, np.int64(2): 19671, np.int64(3): 19761, np.int64(4): 23298}

Pays sans région tenue à l'écart : 6 ['Côte dIvoire', 'Mozambique', 'Somalia', 'South Sudan', 'Togo', 'Zambia']
Pays sans donnée à partir de 2017 : 23
Sources du test temporel : {'HarvestStat': 10838, 'RCeA': 1723, 'FAO_subnational': 665}


**Constat**

- Le développement compte 105 724 lignes, soit 72,5 % du jeu, réparties sur 539 groupes et 45 pays, pour les années 1960 à 2016.
- Le test temporel compte 13 226 lignes et 22 pays, le test spatial 23 817 lignes et 39 pays, le test spatio-temporel 3 110 lignes et 17 pays.
- Aucun groupe n'est présent des deux côtés et chaque groupe appartient à un seul pli. Les plis comptent de 17 347 à 25 647 lignes.
- 23 pays n'ont aucune donnée à partir de 2017. 6 pays n'ont aucune région tenue à l'écart, à savoir la Côte d'Ivoire, le Mozambique, la Somalie, le Soudan du Sud, le Togo et la Zambie.
- Le test temporel provient de `HarvestStat` pour 10 838 de ses 13 226 lignes.

**Analyse**

Les plis sont de taille inégale, car un groupe correspond à une région entière et certaines régions concentrent beaucoup de cultures et d'années. Cette inégalité est le prix de l'étanchéité. Le test temporel ne couvre que la moitié des pays, faute de données récentes, et décrira donc surtout les pays suivis par `HarvestStat`. Les pays sans région tenue à l'écart comptent peu de régions, et l'empreinte n'en a retenu aucune.

**Conséquences pour la suite**

- Les scores par pays ne seront interprétés que pour les pays présents dans l'ensemble considéré.
- La moyenne non pondérée par pays sera rapportée à côté de la moyenne globale.

### 4. Partition de `survey`

Le groupe réunit, par composantes connexes, les grappes spatiales et les ménages qui partagent des parcelles. Un ménage du panel reste ainsi dans un seul groupe d'une vague à l'autre, même lorsque ses coordonnées changent. À défaut de grappe et de ménage, la zone de niveau 2 puis la ligne elle-même servent de groupe. La période future correspond à la dernière vague des pays qui en comptent au moins deux.

In [5]:
display(bilan_partitions(survey, "Country"))
print("Étanchéité :", controler_etancheite(survey))
print("Lignes par pli :", survey["pli"].value_counts().sort_index().to_dict())
print("Taille des groupes :", survey["groupe"].value_counts().describe()[["mean", "50%", "max"]].round(1).to_dict())
print("Groupes de repli :", survey["groupe"].str.split("=").str[0].value_counts().to_dict())
display(survey.groupby("Country")["partition"].value_counts().unstack(fill_value=0).reindex(columns=ORDRE, fill_value=0))

,lignes,part,groupes,pays
partition,,,,
developpement,82301,0.545,6659,12
test_temporel,39112,0.259,3491,9
test_spatial,20495,0.136,1613,12
test_spatio_temporel,9230,0.061,885,9


Étanchéité : {'groupes présents des deux côtés': 0, 'plis maximum par groupe': 1}
Lignes par pli : {np.int64(0): 15900, np.int64(1): 15697, np.int64(2): 15759, np.int64(3): 17318, np.int64(4): 17627}
Taille des groupes : {'mean': 12.5, '50%': 3.0, 'max': 456.0}


Groupes de repli : {'id_grappe': 141934, 'id_menage': 9188, 'id_ligne': 16}


partition,developpement,test_temporel,test_spatial,test_spatio_temporel
Country,,,,
Benin,61,0,7,0
Côte dIvoire,499,0,107,0
Ethiopia,19754,2184,4812,654
Guinea-Bissau,915,0,135,0
Malawi,24654,14171,6325,2918
Mali,4392,112,1068,29
Niger,5113,6126,952,1285
Nigeria,3007,5978,811,1722
Senegal,340,858,78,166


**Constat**

- Le développement compte 82 301 lignes, soit 54,5 % du jeu, réparties sur 6 659 groupes et 12 pays.
- Le test temporel compte 39 112 lignes, soit 25,9 % du jeu, dans 9 pays. Le test spatial compte 20 495 lignes et le test spatio-temporel 9 230 lignes.
- Aucun groupe n'est présent des deux côtés et chaque groupe appartient à un seul pli. Un groupe compte 3 parcelles en médiane et 456 au plus.
- 141 934 lignes appartiennent à un groupe comportant au moins une grappe, 9 188 à un groupe formé de ménages sans coordonnées, et 16 lignes forment leur propre groupe.
- Le Bénin, la Côte d'Ivoire et la Guinée-Bissau, qui ne comptent qu'une vague, n'ont pas de période future.

**Analyse**

Le test temporel est volumineux, car la dernière vague de plusieurs pays compte beaucoup de parcelles, notamment au Malawi, au Togo et au Niger. Le développement conserve néanmoins plus de la moitié du jeu et couvre les 12 pays.

**Conséquences pour la suite**

Les pays à vague unique ne contribuent qu'à l'évaluation spatiale.

### 5. Rattachement de `cropcut`

Les mesures directes de récolte ont été réalisées dans les grappes de l'enquête éthiopienne. Chaque mesure hérite donc du groupe de sa grappe dans `survey`. Seules les mesures dont la grappe est tenue à l'écart constituent une validation externe indépendante de l'entraînement.

In [6]:
grappes_communes = cropcut["id_grappe"].isin(survey["id_grappe"])
print("Grappes cropcut :", cropcut["id_grappe"].nunique(), "| présentes dans survey :", cropcut.loc[grappes_communes, "id_grappe"].nunique())
print("Mesures dont la grappe est tenue à l'écart :", int(cropcut["grappe_tenue_a_l_ecart"].sum()), "sur", len(cropcut),
      "| grappes :", cropcut.loc[cropcut["grappe_tenue_a_l_ecart"], "id_grappe"].nunique())
print("Par vague :", cropcut.loc[cropcut["grappe_tenue_a_l_ecart"], "Source"].value_counts().to_dict())

Grappes cropcut : 497 | présentes dans survey : 454
Mesures dont la grappe est tenue à l'écart : 2297 sur 11263 | grappes : 99
Par vague : {'ETH2015': 808, 'ETH2018': 586, 'ETH2013b': 509, 'ETH2011': 394}


**Constat**

- 454 des 497 grappes de `cropcut` figurent dans `survey`.
- 2 297 mesures sur 11 263, réparties dans 99 grappes et sur les quatre vagues, appartiennent à une grappe tenue à l'écart.

**Analyse**

Une validation sur l'ensemble des mesures aurait évalué le modèle sur des grappes déjà vues à l'entraînement, ce qui aurait surestimé sa performance.

**Conséquences pour la suite**

La validation externe porte uniquement sur ces 2 297 mesures. `cropcut` n'est jamais utilisé pour l'entraînement.

### 6. Métriques et modèles de référence

**Métriques**

La métrique principale est l'erreur absolue moyenne sur le logarithme népérien du rendement, notée `mae_log`. Elle mesure une erreur relative et reste comparable entre des cultures dont les rendements diffèrent d'un ordre de grandeur. L'erreur relative typique en est l'exponentielle diminuée de un. L'erreur absolue et l'erreur quadratique moyennes en tonnes par hectare, le biais et le coefficient de détermination sont rapportés pour l'interprétation. Le biais logarithmique signale une surestimation ou une sous-estimation systématique.

**Références**

La référence prédit la moyenne géométrique des rendements d'entraînement au niveau le plus fin observé, puis se replie sur des niveaux plus larges. Pour `regional`, les niveaux successifs sont la culture, la saison et la zone, puis la culture, la saison et la région de niveau 1, puis la culture et le pays, puis la culture seule. Pour `survey`, ce sont la culture et la grappe, puis la culture et la zone de niveau 2, puis la culture et la zone de niveau 1, puis la culture et le pays, puis la culture seule. La référence n'utilise aucune variable explicative et constitue le seuil que tout modèle devra dépasser.

In [7]:
def evaluer_reference(df, cible, niveaux):
    developpement = df[df["partition"] == DEVELOPPEMENT]
    validation = []
    for pli in range(NOMBRE_PLIS):
        entrainement = developpement[developpement["pli"] != pli]
        evaluation = developpement[developpement["pli"] == pli]
        prediction, _ = reference_hierarchique(entrainement, evaluation, cible, niveaux)
        validation.append(evaluation.assign(prediction=prediction))
    resultats = {"validation_croisee": pd.concat(validation)}
    for partition in TESTS:
        evaluation = df[df["partition"] == partition]
        prediction, niveau = reference_hierarchique(developpement, evaluation, cible, niveaux)
        resultats[partition] = evaluation.assign(prediction=prediction, niveau_reference=niveau)
    return resultats


def tableau_scores(resultats, cible, pays):
    lignes = {}
    for nom, df in resultats.items():
        scores = metriques(df[cible], df["prediction"])
        scores["mae_log_macro_pays"] = moyenne_macro(metriques_par(df, pays, cible, "prediction"))
        lignes[nom] = scores
    return pd.DataFrame(lignes).T.round(3)


references_regional = evaluer_reference(regional, "yieldTonHa", NIVEAUX_REGIONAL)
scores_regional = tableau_scores(references_regional, "yieldTonHa", "L0_GID")
display(scores_regional)
print("Niveau de la référence utilisé dans chaque test :")
display(pd.DataFrame({nom: references_regional[nom]["niveau_reference"].value_counts(normalize=True).sort_index().round(3) for nom in TESTS}))

,n,mae_log,erreur_relative_typique,mae,rmse,biais,biais_log,r2,mae_log_macro_pays
validation_croisee,105724.0,0.417,0.518,1.123,9.464,-0.434,0.001,0.161,0.411
test_temporel,13226.0,0.398,0.490,1.269,3.767,-0.449,-0.086,0.534,0.432
test_spatial,23817.0,0.388,0.474,1.270,16.048,-0.608,-0.013,0.059,0.390
test_spatio_temporel,3110.0,0.404,0.498,1.380,3.559,-0.585,-0.113,0.485,0.495


Niveau de la référence utilisé dans chaque test :


,test_temporel,test_spatial,test_spatio_temporel
niveau_reference,,,
0,0.896,0.0,<NA>
1,0.041,<NA>,<NA>
2,0.056,0.999,0.992
3,0.007,0.001,0.008


In [8]:
references_survey = evaluer_reference(survey, "YieldUsingGPS_ton_ha_", NIVEAUX_SURVEY)
scores_survey = tableau_scores(references_survey, "YieldUsingGPS_ton_ha_", "Country")
display(scores_survey)

developpement_survey = survey[survey["partition"] == DEVELOPPEMENT]
externe = cropcut[cropcut["grappe_tenue_a_l_ecart"]]
prediction_externe, _ = reference_hierarchique(developpement_survey, externe, "YieldUsingGPS_ton_ha_", NIVEAUX_SURVEY)
scores_cropcut = pd.Series(metriques(externe["Yield_ton_ha_"], prediction_externe)).round(3)
print("Validation externe sur cropcut, grappes tenues à l'écart :")
display(scores_cropcut.to_frame("cropcut").T)

,n,mae_log,erreur_relative_typique,mae,rmse,biais,biais_log,r2,mae_log_macro_pays
validation_croisee,82301.0,0.772,1.164,1.182,3.328,-0.483,0.007,0.161,0.806
test_temporel,39112.0,0.828,1.289,1.218,3.009,-0.483,-0.236,-0.007,0.815
test_spatial,20495.0,0.756,1.130,1.094,3.084,-0.508,0.013,0.185,0.779
test_spatio_temporel,9230.0,0.822,1.275,1.335,3.466,-0.433,-0.158,-0.038,0.817


Validation externe sur cropcut, grappes tenues à l'écart :


,n,mae_log,erreur_relative_typique,mae,rmse,biais,biais_log,r2
cropcut,2297.0,0.676,0.967,0.647,0.907,0.057,0.252,-0.128


In [9]:
sensibilite = {}
for nom, df in references_survey.items():
    conservees = df[~df["flag_surface_gps_recopiee"]]
    sensibilite[nom] = {
        "parcelles retirées": int(df["flag_surface_gps_recopiee"].sum()),
        "mae_log toutes parcelles": metriques(df["YieldUsingGPS_ton_ha_"], df["prediction"])["mae_log"],
        "mae_log sans surface recopiée": metriques(conservees["YieldUsingGPS_ton_ha_"], conservees["prediction"])["mae_log"],
    }
print("Sensibilité de la référence survey aux surfaces GPS recopiées :")
display(pd.DataFrame(sensibilite).T.round(3))

print("Erreur logarithmique de la référence regional par source, validation croisée :")
display(metriques_par(references_regional["validation_croisee"], "Source", "yieldTonHa", "prediction")[["n", "mae_log", "biais_log"]].round(3))

print("Erreur logarithmique de la référence par culture, validation croisée regional, cultures de plus de 2 000 lignes :")
par_culture = metriques_par(references_regional["validation_croisee"], "crop", "yieldTonHa", "prediction")
display(par_culture.loc[par_culture["n"] > 2000, ["n", "mae_log", "biais_log"]].sort_values("mae_log").round(3))

Sensibilité de la référence survey aux surfaces GPS recopiées :


,parcelles retirées,mae_log toutes parcelles,mae_log sans surface recopiée
validation_croisee,1752.0,0.772,0.772
test_temporel,4028.0,0.828,0.835
test_spatial,507.0,0.756,0.756
test_spatio_temporel,848.0,0.822,0.826


Erreur logarithmique de la référence regional par source, validation croisée :


,n,mae_log,biais_log
Source,,,
AgroMAPS,8160,0.323,0.018
FAO_subnational,5065,0.470,-0.249
HarvestStat,86668,0.421,0.018
RCeA,5759,0.444,-0.069
USAID_KadunaState,72,0.417,-0.264


Erreur logarithmique de la référence par culture, validation croisée regional, cultures de plus de 2 000 lignes :


,n,mae_log,biais_log
crop,,,
Beans,5678,0.372,-0.005
Groundnut,9798,0.376,-0.003
Yam,2380,0.386,0.025
Cotton,2797,0.389,0.001
Millet,9034,0.390,0.002
Maize,14894,0.391,-0.002
Sorghum,11060,0.395,-0.002
Sweet potato,5532,0.405,-0.001
Cassava,5778,0.416,-0.000


**Constat**

- Dans `regional`, la référence atteint une erreur logarithmique de 0,417 en validation croisée, soit une erreur relative typique de 51,8 %. Elle atteint 0,398 sur le test temporel, 0,388 sur le test spatial et 0,404 sur le test spatio-temporel.
- Sur le test temporel, la référence s'appuie sur l'historique de la zone pour 89,6 % des lignes. Sur le test spatial, elle se replie sur la moyenne de la culture dans le pays pour 99,9 % des lignes.
- Le biais logarithmique vaut -0,086 sur le test temporel et -0,113 sur le test spatio-temporel.
- En validation croisée, `FAO_subnational` présente un biais logarithmique de -0,249, contre 0,018 pour `HarvestStat`.
- L'erreur quadratique en tonnes par hectare varie de 3,559 à 16,048 selon l'ensemble, alors que l'erreur logarithmique reste comprise entre 0,388 et 0,417.
- Dans `survey`, l'erreur logarithmique de la référence vaut 0,772 en validation croisée, soit une erreur relative typique de 116,4 %, et 0,828 sur le test temporel, avec un biais logarithmique de -0,236. Le coefficient de détermination ne dépasse pas 0,185.
- Sur les 2 297 mesures `cropcut` indépendantes, la référence entraînée sur `survey` atteint une erreur logarithmique de 0,676 et un biais logarithmique de 0,252.
- Retirer les surfaces GPS recopiées ne modifie pas l'erreur en validation croisée et la fait passer de 0,828 à 0,835 sur le test temporel.

**Analyse**

- Dans `regional`, la moyenne de la culture dans le pays prédit une région inconnue presque aussi bien que l'historique de la zone ne prédit une année future. Les variations d'une année à l'autre pèsent donc autant que les différences entre régions d'un même pays, ce qui justifie le recours à des variables climatiques annuelles.
- Le biais négatif sur les périodes futures traduit une hausse des rendements dans le temps, que la référence ignore. Les modèles devront intégrer une tendance temporelle.
- Le biais de `FAO_subnational` confirme le désaccord entre sources établi lors du Data Understanding.
- L'erreur quadratique est dominée par les cultures à rendement élevé. Elle ne peut servir de métrique principale sans privilégier ces cultures.
- À l'échelle de la parcelle, l'erreur de la référence est près de deux fois plus élevée qu'à l'échelle régionale, ce qui reflète le bruit de mesure et l'hétérogénéité des pratiques entre parcelles voisines.
- Le biais positif sur `cropcut` indique que les rendements d'enquête dépassent les mesures directes d'environ 29 %. Un modèle entraîné sur `survey` héritera de ce biais.
- Les surfaces recopiées n'altèrent pas la référence, qui n'utilise pas la surface. Leur effet devra être réévalué avec les modèles.

**Conséquences pour la suite**

- La source devra figurer parmi les variables de `regional` ou, à défaut, dans la ventilation des scores.
- Le biais par rapport à `cropcut` sera rapporté pour chaque modèle et ne sera pas corrigé sans validation spécifique.

### 7. Synthèse et règles du protocole

| Élément | Règle |
|---|---|
| Métrique principale | `mae_log`, erreur absolue moyenne sur le logarithme népérien du rendement |
| Métriques secondaires | Erreur relative typique, erreurs absolue et quadratique en tonnes par hectare, biais, biais logarithmique, coefficient de détermination, moyenne non pondérée par pays |
| Sélection des modèles | Validation croisée à 5 plis groupés, sur le développement uniquement |
| Ensembles de test | Utilisés une seule fois, à la fin de la modélisation, pour les modèles retenus |
| Ventilations | Partition, pays, culture, niveau administratif et source pour `regional` |
| Validation externe | 2 297 mesures `cropcut` des grappes tenues à l'écart, sans aucun entraînement sur `cropcut` |
| Contrôle de sensibilité | Scores de `survey` avec et sans les surfaces GPS recopiées |
| Variables explicatives | Disponibles au moment de la prédiction, jamais issues d'une colonne classée comme fuite dans le dictionnaire. Tout encodage fondé sur la cible est recalculé dans chaque pli d'entraînement |
| Score de compétence | Part de l'erreur `mae_log` de la référence supprimée par le modèle |

**Scores de référence à battre**

| Jeu | Validation croisée | Test temporel | Test spatial | Test spatio-temporel |
|---|---:|---:|---:|---:|
| `regional` | 0,417 | 0,398 | 0,388 | 0,404 |
| `survey` | 0,772 | 0,828 | 0,756 | 0,822 |
| `cropcut`, validation externe | | | 0,676 | |

**Décision**

La troisième condition de la décision issue du Data Understanding est remplie. Le protocole d'évaluation est défini, implémenté et testé avant l'entraînement du premier modèle. Un modèle ne sera retenu que s'il obtient un score de compétence positif en validation croisée et le confirme sur les ensembles de test.

**Conséquences pour la suite**

Les variables explicatives doivent maintenant être choisies et acquises, car les jeux préparés ne contiennent que la cible, la culture, le lieu, la saison et l'année.